In [200]:
import sys
import os
import pandas as pd


In [201]:
sys.path.append('../../one_to_one/')


In [203]:
df_in= pd.read_csv('data/annotations_merged/women_coats_annotation_merged.csv')

In [204]:
df_in.columns

Index(['Unnamed: 0', 'department', 'accents', 'features', 'style',
       'only_filenames', 'closure', 'fabric', 'pattern', 'material',
       'occasion', 'collar_style', 'coat_length'],
      dtype='object')

In [205]:
cols_to_exlude_to_display = ['Unnamed: 0','only_filenames']
attribute_columns = [col for col in df_in.columns if col not in cols_to_exlude_to_display]
attribute_columns

['department',
 'accents',
 'features',
 'style',
 'closure',
 'fabric',
 'pattern',
 'material',
 'occasion',
 'collar_style',
 'coat_length']

In [206]:
df_coats = df_in.loc[df_in['department']=='coat' ]
df_coats=df_coats.drop(columns='Unnamed: 0')

In [216]:
for col in attribute_columns:
    print(col)
    print(df_coats[col].value_counts())

department
coat    12508
Name: department, dtype: int64
accents
['Not-Applicable/Visible']    9245
[]                            1403
['fur_trim']                   902
['quilted']                    714
['fur_trim', 'quilted']        219
['studded']                      6
['logo']                         1
Name: accents, dtype: int64
features
['collared']                                 5199
['collared', 'pockets']                      2694
[]                                           1692
['hooded']                                    970
['pockets']                                   884
['pockets', 'hooded']                         305
['zipped_pockets']                            199
['Not-Applicable/Visible']                    186
['zipped_pockets', 'hooded']                   84
['collared', 'zipped_pockets']                 84
['fishtail', 'hooded']                         27
['pockets', 'zipped_pockets']                  14
['collared', 'pockets', 'hooded']              13
['fi

In [207]:
""" convert multi attribute string object into list object """
def convert_multiattrib_string_into_list(multi_string):
    return multi_string.replace('[','').replace(']','').replace("'",'').replace(' ','').split(',')

In [208]:
""" check if input string is multi attibute string """
def is_multiattrib_string(string):
    if '[' in string and ']' in string:
        return True
    return False

In [209]:
# this method takes single row dictionry 
# returns multiple rows dataframe
# where each row contains image id and single attribute value

def gen_one2one_rows_from_dict(input_dict,image_id_name=None,cols=None):
    if image_id_name is None:
        image_id_name = 'only_filenames'
    
    image_id = input_dict[image_id_name]
    one2one_rows_list = []


    for col in cols:
        dct = {}
        dct[image_id_name]=image_id
        attrib_val = input_dict[col]
        
        
        # deal with string values only
        if isinstance(attrib_val,str):
            
            # process multi attrib values strings
            if is_multiattrib_string(attrib_val):
                if attrib_val=='[]':
                    dct={}
                    dct[image_id_name]=image_id
                    dct[col]=None
                    one2one_rows_list.append(dct)
                else:
                    attrib_vals_list=convert_multiattrib_string_into_list(attrib_val)  
                    for val in attrib_vals_list:
                        dct={}
                        dct[image_id_name]=image_id
                        dct[col]=val
                        one2one_rows_list.append(dct)
            # process single attrib value
            else:
                dct = {}
                dct[image_id_name] = image_id
                dct[col]=attrib_val
                one2one_rows_list.append(dct)
        elif attrib_val is None:
            dct = {}
            dct[image_id_name] = image_id
            dct[col] = attrib_val
            one2one_rows_list.append(dct)

    return pd.DataFrame(one2one_rows_list)


In [212]:

dfs_list = []
for i in range(df_coats.shape[0]):
    dfs_list.append(gen_one2one_rows_from_dict(df_coats.iloc[i].to_dict(),cols=attribute_columns))

In [213]:
dataframe=pd.concat(dfs_list)
dataframe

,only_filenames,department,accents,features,style,fabric,pattern,material,collar_style,coat_length,closure,occasion
0,coat_Style--parka_women_coats--local--Images--...,coat,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,coat_Style--parka_women_coats--local--Images--...,NaN,Not-Applicable/Visible,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,coat_Style--parka_women_coats--local--Images--...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,coat_Style--parka_women_coats--local--Images--...,NaN,NaN,NaN,Not-Applicable/Visible,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,coat_Style--parka_women_coats--local--Images--...,NaN,NaN,NaN,NaN,Not-Applicable/Visible,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
3,coat_Style--varsity_jacket_women_coats--local-...,NaN,NaN,NaN,teddy,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,coat_Style--varsity_jacket_women_coats--local-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,button,NaN
5,coat_Style--varsity_jacket_women_coats--local-...,NaN,NaN,NaN,NaN,fleece,NaN,NaN,NaN,NaN,NaN,NaN
6,coat_Style--varsity_jacket_women_coats--local-...,NaN,NaN,NaN,NaN,NaN,solid,NaN,NaN,NaN,NaN,NaN


In [215]:
for col in attribute_columns:
    print(col)
    print(dataframe[col].value_counts())

department
coat    12508
Name: department, dtype: int64
accents
Not-Applicable/Visible    9245
fur_trim                  1121
quilted                    933
studded                      6
logo                         1
Name: accents, dtype: int64
features
collared                  8011
pockets                   3934
hooded                    1425
zipped_pockets             397
Not-Applicable/Visible     186
fishtail                    45
elastic_waist               18
sheer                        8
Name: features, dtype: int64
style
Not-Applicable/Visible    2464
puffer_jacket              903
teddy                      829
trench_coat                638
leather/faux_jacket        316
blazer_jacket              254
pea_coat                   215
parka                      187
jeans_jacket                58
snow/ski_jacket             53
vest                        10
cape                         8
Name: style, dtype: int64
closure
Not-Applicable/Visible    3925
button                  